### Задание 3 - 30 баллов

1. Построить более сложную модель с подбором гиперпараметров

В рамках данного пункта необходимо использовать более сложную модель для решения задачи,  оптимизировать гиперпараметры и оценить ее качество.

Критерии оценки:
- Выбрана более сложная ML-модель - **1 балл**
- Произведен подбор гиперпараметров с scikit-learn методами на кросс-валидации  - **2 балла**
- Произведен подбор гиперпараметров с optuna на кросс-валидации  - **3 балла**
- Выбранная модель обучена с лучшими подобранными значениями гиперпараметров - **2 балла**
- Произведено измерение качества на отложенной выборке с использованием ранее выбранной метрики - **1 балл**

2. Проинтерпретировать полученную модель

В рамках данного пункта необходимо проинтерпретировать модель, полученную в предыдущем пункте.

Критерии оценки:
- Получена интерпретация построенной модели, включая визуализации (коэффициенты/permutation importances/shap и тд) - **4 баллов**
- Приведено экспертное мнение о полученной интерпретации (вы, как эксперт в предметной области, можете оценить адекватность признаков и решений, принимаемых моделью, и выразить свое мнение в 1-2 предложении) - **4 баллов**
  
Общее

- Обеспечена воспроизводимость решения: зафиксированы `random_state`, ноутбук воспроизводится от начала до конца без ошибок - **3 балла**
- Соблюден code style на уровне [pep8](https://peps.python.org/pep-0008/) и [On writing clean Jupyter notebooks](https://ploomber.io/blog/clean-nbs/) - **4 балла**
- Принимаемые решения обоснованы и прокомментированы в markdown ячейках (например, если выбран градиентный бустинг, то какие гиперпараметры оптимизирутся, и почему именно они и тп) - **6 баллов**

In [1]:
# Импортируем библиотеки
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,MinMaxScaler,StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score,classification_report

!pip install optuna
import optuna

1) Загружаем датасет из предыдущего задания
---



In [2]:
df = pd.read_csv('HR-data.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,2,1102,2,1,2,1,2,0,...,3,1,0,8,0,1,6,4,0,5
1,49,0,1,279,1,8,1,1,3,1,...,4,4,1,10,3,3,10,7,1,7
2,37,1,2,1373,1,2,2,4,4,1,...,3,2,0,7,3,3,0,0,0,0
3,33,0,1,1392,1,3,4,1,4,0,...,3,3,0,8,3,3,8,7,3,0
4,27,0,2,591,1,2,1,3,1,1,...,3,4,1,6,3,3,2,2,2,2


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   int64
 2   BusinessTravel            1470 non-null   int64
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   int64
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   int64
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   int64
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   int64
 14  JobSatisfaction           1470 non-null 

Разбиваем дата сет на train и test



In [4]:
x = df.drop(columns=['Attrition'])
y = df['Attrition']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=8)

Выберем более сложную ML-модель а именно RandomForestClassifier



In [5]:
rf = RandomForestClassifier(random_state=8)
rf.fit(x_train, y_train)
y_preds = rf.predict(x_test)
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 8,
 'verbose': 0,
 'warm_start': False}

Осуществим подбор гиперпараметров с scikit-learn методами на кросс-валидации а именно GridSearchCV, потому что он позволит перебрать возможные варианты наиболее ресурсоемким путем, с учетом использвования модели RandomForestClassifier

In [6]:
parameters = {
    # Ограничим количество ветвей деревьев и установим шаг в 20, для ускорения работы модели и её возможности запуска в условиях малых мощностей
    'n_estimators': np.arange(40, 140, 20),
    'max_depth': list(range(3, 12)) + [None],
    'random_state': [8],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=8),
    param_grid=parameters,
    n_jobs=-1,
    verbose=1,
    cv=3,
)

In [7]:
grid_search.fit(x_train, y_train)

best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(parameters.keys()):
    print(f"{param_name}: {best_parameters[param_name]}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits
max_depth: 10
n_estimators: 80
random_state: 8


Выпишем предложенные параметры модели RandomForestClassifier

`max_depth`: 10
`n_estimators`: 80
`random_state`: 8

In [8]:
y_preds = rf.predict(x_test)
print(classification_report(y_test, y_preds))

#Для оценки модели в качестве выбранной метрики используем Accuracy
rf_accuracy = accuracy_score(y_test, y_preds)
print(f'Accuracy: {rf_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.88      0.99      0.93       251
           1       0.73      0.19      0.30        43

    accuracy                           0.87       294
   macro avg       0.80      0.59      0.61       294
weighted avg       0.85      0.87      0.84       294

Accuracy: 0.8707


Зафиксируем полученные результаты обучения для модели RandomForestClassifier в случае использования метода GridSearchCV

Accuracy: 0.8707
В сравнении с Accuracy: 0.8673 полученой при использовании DecisionTreeClassifier

Произведем подбор гиперпараметров с optuna на кросс-валидации

In [9]:
def chose(trial: optuna.Trial):
    max_depth = trial.suggest_int('max_depth', 2, 26, log=True)
    n_estimators = trial.suggest_int('n_estimators', 2, 100)
    max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])

    classifier_rf = RandomForestClassifier(
        max_depth=max_depth,
        n_estimators=n_estimators,
        max_features=max_features,
        random_state=8
    )

    # Кросс-валидация с возвратом средней точности
    score = cross_val_score(classifier_rf, x_train, y_train, cv=3, n_jobs=-1)
    return score.mean()  # Возвращаем среднюю точность кросс-валидации

In [10]:
sampler = optuna.samplers.TPESampler(seed=8)
studyrf = optuna.create_study(direction="maximize", sampler=sampler)
studyrf.optimize(chose, n_trials=50)

[I 2024-10-31 14:21:30,483] A new study created in memory with name: no-name-6923a911-0c58-4b44-8a33-36cf06d3201a
[I 2024-10-31 14:21:32,679] Trial 0 finished with value: 0.8452380952380952 and parameters: {'max_depth': 18, 'n_estimators': 97, 'max_features': None}. Best is trial 0 with value: 0.8452380952380952.
[I 2024-10-31 14:21:32,913] Trial 1 finished with value: 0.8358843537414966 and parameters: {'max_depth': 2, 'n_estimators': 44, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8452380952380952.
[I 2024-10-31 14:21:33,763] Trial 2 finished with value: 0.8503401360544217 and parameters: {'max_depth': 7, 'n_estimators': 55, 'max_features': None}. Best is trial 2 with value: 0.8503401360544217.
[I 2024-10-31 14:21:34,149] Trial 3 finished with value: 0.848639455782313 and parameters: {'max_depth': 5, 'n_estimators': 30, 'max_features': None}. Best is trial 2 with value: 0.8503401360544217.
[I 2024-10-31 14:21:34,652] Trial 4 finished with value: 0.835034013605442 and param

In [11]:
print(studyrf.best_trial)

FrozenTrial(number=31, state=TrialState.COMPLETE, values=[0.8554421768707483], datetime_start=datetime.datetime(2024, 10, 31, 14, 21, 46, 935266), datetime_complete=datetime.datetime(2024, 10, 31, 14, 21, 47, 248745), params={'max_depth': 18, 'n_estimators': 35, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=26, log=True, low=2, step=1), 'n_estimators': IntDistribution(high=100, log=False, low=2, step=1), 'max_features': CategoricalDistribution(choices=(None, 'sqrt', 'log2'))}, trial_id=31, value=None)


 Лучшие параметры

params= {`'max_depth': 18`, `'n_estimators': 35`, `'max_features': 'sqrt`'}

In [12]:
studyrf.best_params

{'max_depth': 18, 'n_estimators': 35, 'max_features': 'sqrt'}

In [13]:
Rforest = RandomForestClassifier(random_state=8,**studyrf.best_params)
Rforest.fit(x_train, y_train)

y_preds = Rforest.predict(x_test)
print(classification_report(y_test, y_preds))

#Оцениваем качество модели по метрике F1
Rforest_accuracy = accuracy_score(y_test, y_preds)
print(f'Accuracy: {Rforest_accuracy:.4f}')

              precision    recall  f1-score   support

           0       0.87      0.99      0.93       251
           1       0.70      0.16      0.26        43

    accuracy                           0.87       294
   macro avg       0.79      0.58      0.60       294
weighted avg       0.85      0.87      0.83       294

Accuracy: 0.8673


In [14]:
from optuna.visualization import plot_optimization_history

plot_optimization_history(studyrf)

In [15]:
from optuna.visualization import plot_slice

plot_slice(studyrf)